# 4.1 Creating and managing tensors

In [4]:
import torch
import numpy as np

# 1. Creating Tensors
print("1. Creating Tensors:")

# From Python list
tensor_from_list = torch.tensor([1, 2, 3, 4])
print("Tensor from list:", tensor_from_list)

# From NumPy array
np_array = np.array([1, 2, 3, 4])
tensor_from_np = torch.from_numpy(np_array)
print("Tensor from NumPy array:", tensor_from_np)

# Random tensor
random_tensor = torch.randn(3, 4)
print("Random Tensor:", random_tensor)

# 2. Basic Operations
print("2. Basic Operations:")

# Element-wise operations
a = torch.tensor([1, 2, 3])
b = torch.tensor([4, 5, 6])
print("Addition:", a + b)
print("Multiplication:", a * b)

# Reduction operations
tensor_sum = torch.sum(random_tensor)
tensor_mean = torch.mean(random_tensor)
print(f"Sum of tensor elements: {tensor_sum.item()}")
print(f"Mean of tensor elements: {tensor_mean.item()}")

# 3. Reshaping Tensors
print("3. Reshaping Tensors:")
c = torch.tensor([[1, 2, 3, 4], [5, 6, 7, 8]])
print("Original shape:", c.shape)
reshaped = c.reshape(4, 2)
print("Reshaped:", reshaped)

# 4. Indexing and Slicing
print("4. Indexing and Slicing:")
print("First row:", c[0])
print("Second column:", c[:, 1])

# 5. GPU Operations
print("5. GPU Operations:")
if torch.cuda.is_available():
    gpu_tensor = torch.zeros(3, 4, device='cuda')
    print("Tensor on GPU:", gpu_tensor)
    # Move tensor to CPU
    cpu_tensor = gpu_tensor.to('cpu')
    print("Tensor moved to CPU:", cpu_tensor)
else:
    print("CUDA is not available. Using CPU instead.")
    cpu_tensor = torch.zeros(3, 4)
    print("Tensor on CPU:", cpu_tensor)
    
# 6. Autograd (Automatic Differentiation)
print("6. Autograd:")
x = torch.tensor([2.0], requires_grad=True)
y = x ** 2
y.backward()
print("Gradient of y=x^2 at x=2:", x.grad)

1. Creating Tensors:
Tensor from list: tensor([1, 2, 3, 4])
Tensor from NumPy array: tensor([1, 2, 3, 4])
Random Tensor: tensor([[-0.3168,  0.5575, -1.1159, -0.3468],
        [-0.6379, -0.6661,  1.0194, -0.5490],
        [-0.4163, -0.6031, -0.2603,  1.3229]])
2. Basic Operations:
Addition: tensor([5, 7, 9])
Multiplication: tensor([ 4, 10, 18])
Sum of tensor elements: -2.0121777057647705
Mean of tensor elements: -0.1676814705133438
3. Reshaping Tensors:
Original shape: torch.Size([2, 4])
Reshaped: tensor([[1, 2],
        [3, 4],
        [5, 6],
        [7, 8]])
4. Indexing and Slicing:
First row: tensor([1, 2, 3, 4])
Second column: tensor([2, 6])
5. GPU Operations:
CUDA is not available. Using CPU instead.
Tensor on CPU: tensor([[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]])
6. Autograd:
Gradient of y=x^2 at x=2: tensor([4.])


# 4.2 Definition of a simple computational graph

In [ ]:
import torch

# Create tensors with gradient tracking enabled
x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(3.0, requires_grad=True)

# Define a more complex computation
z = x**2 + 2 * x * y + y**2
print(f"z = {z.item()}")

# Perform backpropagation to compute the gradients
z.backward()

# Print the gradients (derivatives of z w.r.t. x and y)
print(f"Gradient of z with respect to x: {x.grad.item()}")
print(f"Gradient of z with respect to y: {y.grad.item()}")

# Reset gradients
x.grad.zero_()
y.grad.zero_()

# Define another computation
w = torch.log(x) + torch.exp(y)
print(f"w = {w.item()}")

# Compute gradients for w
w.backward()

# Print the new gradients
print(f"Gradient of w with respect to x: {x.grad.item()}")
print(f"Gradient of w with respect to y: {y.grad.item()}")

# Demonstrate higher-order gradients
x = torch.tensor(2.0, requires_grad=True)
y = x**2 + 2 * x + 1

# Compute first-order gradient
first_order = torch.autograd.grad(y, x, create_graph=True)[0]
print(f"First-order gradient: {first_order.item()}")

# Compute second-order gradient
second_order = torch.autograd.grad(first_order, x)[0]
print(f"Second-order gradient: {second_order.item()}")

z = 25.0
Gradient of z with respect to x: 10.0
Gradient of z with respect to y: 10.0
w = 20.778684616088867
Gradient of w with respect to x: 0.5
Gradient of w with respect to y: 20.08553695678711
First-order gradient: 6.0
Second-order gradient: 2.0


# 4.3 Definition of a feedforward neural network

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchinfo import summary
# Define a neural network by subclassing nn.Module

class ComprehensiveNN(nn.Module):
    def __init__(self, input_size, hidden_sizes, output_size, dropout_rate=0.5):
        super(ComprehensiveNN, self).__init__()
        self.input_size = input_size
        self.hidden_sizes = hidden_sizes
        self.output_size = output_size
        # Create a list of linear layers
        self.hidden_layers = nn.ModuleList()
        all_sizes = [input_size] + hidden_sizes
        for i in range(len(all_sizes)-1):
            self.hidden_layers.append(nn.Linear(all_sizes[i], all_sizes[i+1]))
        
        # Output layer
        self.output_layer = nn.Linear(hidden_sizes[-1], output_size)
        # Dropout layer
        self.dropout = nn.Dropout(dropout_rate)
        # Batch normalization layers
        self.batch_norms = nn.ModuleList([nn.BatchNorm1d(size) for size in hidden_sizes])
        
    def forward(self, x):
        # Flatten the input tensor
        x = x.view(-1, self.input_size)
        # Apply hidden layers with ReLU, BatchNorm, and Dropout
        for i, layer in enumerate(self.hidden_layers):
            x = layer(x)
            x = self.batch_norms[i](x)
            x = F.relu(x)
            x = self.dropout(x)
            
        # Output layer (no activation for use with CrossEntropyLoss)
        x = self.output_layer(x)
        return x
    
# Hyperparameters
input_size = 784  # 28×28 MNIST images
hidden_sizes = [256, 128, 64]
output_size = 10  # 10 digit classes
learning_rate = 0.001
batch_size = 64
num_epochs = 10
# Instantiate the model
model = ComprehensiveNN(input_size, hidden_sizes, output_size)
print(model)
summary(model, input_size=(1, 784))
# Load and preprocess the MNIST dataset
transform = transforms.Compose(
    [transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))]
)

train_dataset = torchvision.datasets.MNIST(
    root="./data", train=True, download=True, transform=transform
)
test_dataset = torchvision.datasets.MNIST(
    root="./data", train=False, download=True, transform=transform
)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

# Training loop
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for i, (images, labels) in enumerate(train_loader):
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    running_loss += loss.item()
    
print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}')

# Evaluation
model.eval()

print(
    summary(
        model,
        input_size=(1, 784),
        depth=4,
        col_names=("input_size", "output_size", "num_params"),
        verbose=0,
    )
)

with torch.no_grad():
    correct = 0
    total = 0
    for images, labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
print(f'Accuracy on the test set: {100 * correct / total:.2f}%')

ComprehensiveNN(
  (hidden_layers): ModuleList(
    (0): Linear(in_features=784, out_features=256, bias=True)
    (1): Linear(in_features=256, out_features=128, bias=True)
    (2): Linear(in_features=128, out_features=64, bias=True)
  )
  (output_layer): Linear(in_features=64, out_features=10, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (batch_norms): ModuleList(
    (0): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  )
)
Epoch [10/10], Loss: 0.0004
Layer (type:depth-idx)                   Input Shape               Output Shape              Param #
ComprehensiveNN                          [1, 784]                  [1, 10]                   --
├─ModuleList: 1-7                        --                        --                    